In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. Carrega as variáveis de ambiente (as mesmas usadas no load.py)
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5433") # Lembre-se que configuramos o Docker na 5433!
DB_NAME = os.getenv("DB_NAME", "olist_db")

# 2. Cria o motor de conexão com o PostgreSQL
string_conexao = f"postgresql+psycopg://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(string_conexao)

print("✅ Conectado ao banco de dados local com sucesso!")

✅ Conectado ao banco de dados local com sucesso!


In [2]:
# Consulta o catálogo interno do PostgreSQL para listar as tabelas criadas
query_tabelas = """
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_name;
"""

df_tabelas = pd.read_sql(query_tabelas, con=engine)

print("🗄️ TABELAS ENCONTRADAS NO BANCO DE DADOS:")
display(df_tabelas)

🗄️ TABELAS ENCONTRADAS NO BANCO DE DADOS:


,table_name
0,dim_clientes
1,dim_produtos
2,dim_vendedores
3,fato_itens
4,fato_pagamentos
5,fato_pedidos


In [3]:
print("📊 AUDITORIA DE VOLUME (TOTAL DE REGISTROS POR TABELA):")

tabelas_lista = df_tabelas['table_name'].tolist()
auditoria_volume = []

# Loop executando um COUNT(*) dinâmico para cada tabela
for tabela in tabelas_lista:
    query_count = f"SELECT COUNT(*) AS total_linhas FROM {tabela}"
    total = pd.read_sql(query_count, con=engine).iloc[0, 0]
    
    auditoria_volume.append({'Tabela': tabela, 'Total de Registros': total})

# Transforma o resultado em um DataFrame para visualização limpa
df_volume = pd.DataFrame(auditoria_volume)
display(df_volume)

📊 AUDITORIA DE VOLUME (TOTAL DE REGISTROS POR TABELA):


,Tabela,Total de Registros
0,dim_clientes,99441
1,dim_produtos,32951
2,dim_vendedores,3095
3,fato_itens,112650
4,fato_pagamentos,103886
5,fato_pedidos,99441


In [4]:
# Puxa o dataset inteiro de pedidos para checagem rápida de nulos e tipos
query_pedidos = "SELECT * FROM fato_pedidos"
df_teste_pedidos = pd.read_sql(query_pedidos, con=engine)

print("--- 🔎 VERIFICAÇÃO DE TIPOS DE DADOS (SCHEMA NO BANCO) ---")
df_teste_pedidos.info()

print("\n--- ⚠️ CHECAGEM DE INTEGRIDADE (NULOS NAS CHAVES) ---")
# Valida se existe algum pedido sem ID ou sem Cliente (não deve existir nenhum)
nulos_chaves = df_teste_pedidos[['order_id', 'customer_id']].isnull().sum()
display(nulos_chaves.to_frame("Valores Nulos"))

--- 🔎 VERIFICAÇÃO DE TIPOS DE DADOS (SCHEMA NO BANCO) ---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB

--- ⚠️ CHECAGEM DE INTEGRIDADE (NULOS NAS CHAVES) ---


,Valores Nulos
order_id,0
customer_id,0


In [6]:
engine.dispose() # Fecha a conexão com o banco de dados
print("✅ Conexão com o banco de dados encerrada com sucesso!")

✅ Conexão com o banco de dados encerrada com sucesso!
